# QICK loopback on the AntSDR E200

Fires a pulse from the QICK signal generator into the AD9361 transmitter, and
captures it on the QICK readout through the external TX -> 10 dB attenuator ->
RX loop. This is the first end-to-end test of the QICK datapath: tProcessor v2
sequencing a generator and a readout, with the averaging buffer read back over
DMA.

Run this notebook as **root** (PYNQ needs to program the PL). Order matters in
two places, both noted where they occur:

* the overlay is loaded **before** the radio is configured, because programming
  the PL resets `axi_ad9361` and the driver re-establishes the sample rate;
* the sample rate is **not** changed afterwards, because `QickSocE200` reads it
  once at load time to build the frequency plan. The third cell asserts that
  software and hardware still agree.

The transmit mux defaults to the stock ADI DMA path, so nothing radiates from
QICK until cell 4 asks for it, and the last cell puts it back.


In [ ]:
%matplotlib inline
import sys, time
import numpy as np
import matplotlib.pyplot as plt

QICK_DIR = '/home/xilinx/qick_e200'
sys.path.insert(0, QICK_DIR)

from qick.ad9361 import QickSocE200
from qick.asm_v2 import AveragerProgramV2

# fs is deliberately not passed: it is read back from the AD9361 after the PL is
# programmed, which is the only moment the value is trustworthy.
#
# QickSocE200 loads a device tree overlay by default (the same pl.dtbo the
# board's /boot/boot.py uses) and restarts iiod afterwards. Both are required:
# programming the PL resets axi_ad9361 under the bound driver stack, and without
# the re-probe the radio is left asleep with a halved sample rate and cannot be
# woken again short of a reboot.
soc = QickSocE200(QICK_DIR + '/qick_e200.bit')
soccfg = soc
print(soc)


In [ ]:
# --- radio ON, and the settings that make a wired loopback measurable --------
# These are the values the base loopback demo established. Sample rate is left
# alone on purpose (see the note at the top).
import adi

LO_HZ  = int(2.40e9)   # same for TX and RX so baseband frequencies line up
BW_HZ  = int(18e6)

sdr = adi.ad9361(uri='local:')

ensm_on_entry = sdr._ctrl.attrs['ensm_mode'].value
sdr._ctrl.attrs['ensm_mode'].value = 'fdd'
print(f"ensm_mode: {ensm_on_entry} -> {sdr._ctrl.attrs['ensm_mode'].value}  (radio on)")

sdr.rx_lo           = LO_HZ
sdr.tx_lo           = LO_HZ
sdr.rx_rf_bandwidth = BW_HZ
sdr.tx_rf_bandwidth = BW_HZ
sdr.rx_enabled_channels = [0]
sdr.tx_enabled_channels = [0]

# Manual gain: an AGC would ride the level and make the result meaningless.
sdr.gain_control_mode_chan0 = 'manual'
sdr.rx_hardwaregain_chan0   = 20.0
sdr.tx_hardwaregain_chan0   = -20.0   # dB of attenuation, 0 = max output

fs_hw = sdr.sample_rate / 1e6
print(f"AD9361 sample rate: {fs_hw:.6f} MHz")
print(f"QICK  refclk_freq : {soc['refclk_freq']:.6f} MHz")
assert abs(fs_hw - soc['refclk_freq']) < 1e-6, (
    'the AD9361 rate has moved since the overlay was loaded, so the QICK '
    'frequency plan is stale -- re-run the load cell')
print('software and hardware agree on the sample rate')


In [ ]:
# --- hand the transmitter to QICK -------------------------------------------
# Until now the DAC has been fed by the stock ADI DMA/DDS path.
print('tx source before:', soc.get_tx_source())
soc.tx_source('qick')
print('tx source now   :', soc.get_tx_source())


In [ ]:
# --- the program ------------------------------------------------------------
# One constant-envelope pulse, and one readout window that opens with it. The
# readout mixes the tone down to DC, so a correct capture is a flat-topped
# envelope with a slowly rotating phase (the residual frequency error).

class LoopbackProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        # axis_readout_v2 is not driven by the tProcessor (its tproc_ch is None),
        # so QICK calls it "static": the downconversion frequency is written by
        # software here rather than sent as a readoutconfig from the program.
        # gen_ch ties the two frequency grids together so the tone lands exactly
        # at the readout's DC.
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'],
                             freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.add_pulse(ch=cfg['gen_ch'], name='probe', ro_ch=cfg['ro_ch'],
                       style='const', freq=cfg['freq'], phase=0,
                       gain=cfg['gain'], length=cfg['pulse_len'])

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name='probe', t=0)
        self.trigger(ros=[cfg['ro_ch']], t=cfg['trig_t'])

f_dec = soc['readouts'][0]['f_output']        # decimated sample rate, MHz
config = dict(
    gen_ch    = 0,
    ro_ch     = 0,
    freq      = 1.0,     # MHz of baseband offset; DDS range is +/- fs/2
    gain      = 0.5,
    pulse_len = 10.0,    # us
    ro_len    = 30.0,    # us -> about %d decimated samples
    trig_t    = 0.0,
)
print(f"decimated rate {f_dec:.3f} MHz -> {config['ro_len']*f_dec:.0f} samples "
      f"in a {config['ro_len']:.0f} us window")


In [ ]:
# --- fire it and capture ----------------------------------------------------
prog = LoopbackProgram(soccfg, reps=1, final_delay=1.0, cfg=config)
iq = prog.acquire_decimated(soc, rounds=10, progress=False)

# acquire_decimated returns one array per readout, shaped (nsamples, 2)
d = iq[0]
i, q = d[:, 0], d[:, 1]
t = np.arange(len(i)) / f_dec        # us
mag = np.abs(i + 1j*q)

print(f"captured {len(i)} decimated samples over {t[-1]:.1f} us")
print(f"peak |IQ| = {mag.max():.1f} ADU at t = {t[np.argmax(mag)]:.2f} us")

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(t, i, '.-', label='I')
ax[0].plot(t, q, '.-', label='Q')
ax[0].set_ylabel('ADU'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[0].set_title('QICK readout, decimated: pulse through the external loopback')
ax[1].plot(t, mag, '.-', color='k')
ax[1].axvspan(0, config['pulse_len'], color='tab:orange', alpha=.15,
              label='commanded pulse')
ax[1].set_xlabel('time (us)'); ax[1].set_ylabel('|IQ| (ADU)')
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()


In [ ]:
# --- is it the right signal? ------------------------------------------------
# Inside the pulse the readout should have mixed the tone to near DC, so the
# residual tone in the decimated data tells us the frequency error. Anything
# near the full decimated bandwidth would mean the readout is demodulating at
# the wrong frequency, or the pulse is not the thing we are seeing.

inside = (t > 1.0) & (t < config['pulse_len'] - 1.0)
seg = (i + 1j*q)[inside]
if len(seg) < 8:
    print('pulse window too short to estimate frequency; lengthen pulse_len')
else:
    win = np.hanning(len(seg))
    sp = np.fft.fftshift(np.fft.fft(seg * win))
    fx = np.fft.fftshift(np.fft.fftfreq(len(seg), d=1.0/f_dec))
    f_err = fx[np.argmax(np.abs(sp))]
    # phase slope is a finer estimate than the FFT bin at this record length
    ph = np.unwrap(np.angle(seg))
    slope = np.polyfit(np.arange(len(seg))/f_dec, ph, 1)[0] / (2*np.pi)
    print(f"residual tone: {f_err*1e3:+.1f} kHz (FFT bin, {f_dec/len(seg)*1e3:.1f} kHz resolution)")
    print(f"               {slope*1e3:+.3f} kHz (phase slope)")
    print(f"in-pulse |IQ|: mean {np.abs(seg).mean():.1f}, "
          f"ripple {100*(np.abs(seg).max()-np.abs(seg).min())/np.abs(seg).mean():.1f} %")

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(fx*1e3, 20*np.log10(np.abs(sp)/np.abs(sp).max() + 1e-12), '.-')
    ax.set_xlabel('offset from readout frequency (kHz)')
    ax.set_ylabel('dB (rel. peak)')
    ax.set_title('Spectrum inside the pulse: a correct demodulation sits at DC')
    ax.grid(alpha=.3)
    plt.tight_layout()


In [ ]:
# --- put everything back ----------------------------------------------------
# Transmit returns to the stock DMA path and the radio is parked in ALERT, which
# keeps the PLLs locked but powers down the signal paths (measured 27 C cooler
# on the AD9361 die, 7 C on the Zynq).
soc.tx_source('dma')
print('tx source restored to:', soc.get_tx_source())

sdr.tx_hardwaregain_chan0 = -89.75
try:
    sdr.tx_hardwaregain_chan1 = -89.75
except Exception:
    pass
sdr._ctrl.attrs['ensm_mode'].value = 'alert'
print('ensm_mode now:', sdr._ctrl.attrs['ensm_mode'].value, '(radio off)')
del sdr
print()
print('Note the PL still holds the QICK overlay, not the boot-time base design.')
print('Reboot to return the board to its stock state.')
